# 01 · Download, sample and preprocess

1. Download `LA.zip` (7.1 GB) from Edinburgh DataShare and extract it (skipped if already present).
2. Parse the official protocol files into `audio_metadata.csv`.
3. Draw the stratified, seeded study sample (synopsis Table 5) → `split_assignments.csv`.
4. Standardise every sampled clip (trim silence, 4 s, RMS) and create telephone copies of the test clips.
5. Quality checks, including the **silence shortcut check**.

Runtime on Colab: download ≈ 10–20 min; preprocessing ≈ 10–15 min.

In [ ]:
# --- Setup: works on Google Colab and locally -------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("VS_DATA_ROOT", "/content/drive/MyDrive/audio-detection-human-vs-ai")
    if not os.path.exists("/content/audio-detection-human-vs-ai"):
        !git clone -q https://github.com/abhilash-u/audio-detection-human-vs-ai /content/audio-detection-human-vs-ai
    %cd /content/audio-detection-human-vs-ai
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
from src import config as C
C.ensure_dirs()
print("Data folder:", C.DATA_ROOT)

In [ ]:
from src import data
zpath = data.download_la()
meta = data.build_metadata()
meta.to_csv(C.META_DIR / "audio_metadata.csv", index=False)
inv = data.write_inventory(zpath, meta, C.META_DIR / "dataset_inventory.csv")
display(inv.T)
meta.groupby(["official_partition", "key"]).size().unstack()

## Draw the study sample

In [ ]:
sample = data.draw_sample(meta)
pd.crosstab(sample.split, sample.label.map({0: "human", 1: "synthetic"}), margins=True)

## Standardise audio and create telephone copies (test clips only)

In [ ]:
from src import preprocess as P
sample = P.process_sample(sample)
sample["channel_available"] = np.where(sample.split.str.startswith("test"), "clean+telephone", "clean")
sample.to_csv(C.META_DIR / "split_assignments.csv", index=False)
sample[["duration_raw_s", "leading_silence_s", "trailing_silence_s", "duration_trimmed_s"]].describe().round(3)

## Quality checks
Balance by split, system and speaker, and duration distributions by class.

In [ ]:
display(sample.groupby(["split", "attack_system_id"]).size().unstack(fill_value=0))
display(sample.groupby("split").speaker_id.nunique().rename("speakers"))
sample.groupby("label")[["duration_raw_s", "leading_silence_s", "trailing_silence_s"]].mean().round(3)

## Shortcut check
A classifier that sees **only** silence lengths should be strong on the raw audio and fall to about 0.5 AUC on the processed clips. If it does not, revise `TRIM_TOP_DB` in `src/config.py` and re-run.

In [ ]:
from src.evaluation import shortcut_auc
proc = P.processed_silence(sample.clip_id.tolist())
s2 = sample.merge(proc, on="clip_id")
tr, va = s2[s2.split == "training"], s2[s2.split == "validation"]
raw_auc = shortcut_auc(tr, va, ["duration_raw_s", "leading_silence_s", "trailing_silence_s"])
proc_auc = shortcut_auc(tr, va, ["proc_leading_silence_s", "proc_trailing_silence_s"])
print(f"Silence-only classifier AUC   raw audio: {raw_auc:.3f}   processed audio: {proc_auc:.3f}")
pd.DataFrame([{"check": "silence shortcut", "auc_raw": raw_auc, "auc_processed": proc_auc,
               "pass": abs(proc_auc - 0.5) < 0.1}]).to_csv(C.REPORT_DIR / "qc_shortcut_check.csv", index=False)